In [22]:
!pip install langchain langgraph langchain-community langchain-ollama faiss-cpu sqlalchemy tavily-python

In [23]:
from typing import TypedDict
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from sqlalchemy import create_engine, text
from langgraph.graph import StateGraph, END
from tavily import TavilyClient

In [24]:
llm = ChatOllama(model="llama3")
embeddings = OllamaEmbeddings(model="nomic-embed-text")

In [25]:
engine = create_engine("sqlite:///example.db")

with engine.connect() as conn:
    conn.execute(text("""
    CREATE TABLE IF NOT EXISTS users (
        id INTEGER PRIMARY KEY,
        name TEXT,
        age INTEGER
    )
    """))

    conn.execute(text("DELETE FROM users"))

    conn.execute(text("""
    INSERT INTO users (name, age) VALUES
    ('Ravi', 30),
    ('Anil', 25),
    ('Sita', 28)
    """))

In [26]:
docs = [
    Document(page_content="Kubernetes is used for container orchestration."),
    Document(page_content="LangGraph is used for agent workflows.")
]

vectorstore = FAISS.from_documents(docs, embeddings)

In [27]:
# YOUR_API_KEY = "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"

client = TavilyClient(api_key="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")  # replace

def web_agent(state):
    query = state["query"]
    print("→ Web Agent")

    results = client.search(query=query)
    content = " ".join([r["content"] for r in results["results"]])

    response = llm.invoke(f"Summarize:\n{content}")

    return {"result": response.content}

In [28]:
def sql_agent(state):
    query = state["query"]
    print("→ SQL Agent")

    prompt = f"You are a SQLite expert. Convert natural language to SQLite-compatible SQL.\\n\\nDatabase schema:\\n- users (id, name, age)\\n\\nRules:\\n- Output ONLY the SQL statement\\n- No markdown, no code fences, no backticks\\n- Use SQLite syntax only\\n\\nQuery: {query}"

    sql = llm.invoke(prompt).content.strip()
    sql = sql.replace("```sql", "").replace("```", "").strip()
    print("Generated SQL:", sql)

    with engine.connect() as conn:
        result = conn.execute(text(sql))
        rows = result.fetchall()

    return {"result": str(rows)}

In [29]:
def rag_agent(state):
    query = state["query"]
    print("→ RAG Agent")

    docs = vectorstore.similarity_search(query)
    context = "\n".join([d.page_content for d in docs])

    prompt = f"""
Answer using:
{context}

Q: {query}
"""
    response = llm.invoke(prompt)

    return {"result": response.content}

In [30]:
def router(state):
    query = state["query"]

    prompt = f"""
Classify the query into EXACTLY one word from this list:
web, sql, rag

Rules:
- Return ONLY one word
- No explanation
- No punctuation

Query: {query}
"""

    raw = llm.invoke(prompt).content.strip().lower()

    print("RAW LLM OUTPUT:", raw)

    # 🔥 HARD SANITIZATION
    if "web" in raw:
        route = "web"
    elif "sql" in raw:
        route = "sql"
    elif "rag" in raw:
        route = "rag"
    else:
        route = "rag"  # fallback

    print("FINAL ROUTE:", route)

    return {"route": route}

In [31]:
class AgentState(TypedDict):
    query: str
    route: str
    result: str

graph = StateGraph(AgentState)

graph.add_node("router", router)
graph.add_node("web", web_agent)
graph.add_node("sql", sql_agent)
graph.add_node("rag", rag_agent)

graph.add_conditional_edges(
    "router",
    lambda x: x["route"],
    {"web": "web", "sql": "sql", "rag": "rag"}
)

graph.add_edge("web", END)
graph.add_edge("sql", END)
graph.add_edge("rag", END)

graph.set_entry_point("router")

app_graph = graph.compile()

In [32]:
def run_query(q):
    result = app_graph.invoke({"query": q})
    print("\nFinal Result:\n", result["result"])

In [33]:
app_graph = graph.compile()
run_query("what is your name?")

RAW LLM OUTPUT: sql
FINAL ROUTE: sql
→ SQL Agent
Generated SQL: SELECT name FROM users;

Final Result:
 []


In [34]:
import subprocess

# First, ensure the model is available by pulling it
subprocess.run(["ollama", "pull", "llama3"], check=False)

# Then run your query
def router(state):
    query = state["query"]

    prompt = f"""
Classify the query into exactly one of: web, sql, rag.
Respond with only one word: web, sql, or rag.

Query: {query}
"""

    response = llm.invoke(prompt)
    text = response.content.strip().lower()

    route = text.split()[0] if text else ""
    if route not in {"web", "sql", "rag"}:
        for candidate in ("web", "sql", "rag"):
            if candidate in text:
                route = candidate
                break

    if route not in {"web", "sql", "rag"}:
        route = "web"

    print("→ Route:", route)
    return {"route": route}

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest 
pulling 6a0746a1ec1a: 100% ▕██████████████████▏ 4.7 GB                         
pulling 4fa551d4f938: 100% ▕██████████████████▏  12 KB                         
pulling 8ab4849b038c: 100% ▕██████████████████▏  254 B                         
pulling 577073ffcc6c: 100% ▕██████████████████▏  110 B                         
pulling 3f8eb4da87fa: 100% ▕██████████████████▏  485 B                         
verifying sha256 digest 
writing manifest 
success 


In [38]:
from sqlalchemy import create_engine, text

engine = create_engine("sqlite:///example.db")

with engine.begin() as conn:  # 🔥 IMPORTANT (auto-commit)
    conn.execute(text("""
    CREATE TABLE IF NOT EXISTS users (
        id INTEGER PRIMARY KEY,
        name TEXT,
        age INTEGER
    )
    """))

    conn.execute(text("DELETE FROM users"))

    conn.execute(text("""
    INSERT INTO users (name, age) VALUES
    ('Ravi', 30),
    ('Anil', 25),
    ('Sita', 28)
    """))

In [39]:
app_graph = graph.compile()
run_query("what are the sql tables available?")

RAW LLM OUTPUT: sql
FINAL ROUTE: sql
→ SQL Agent
Generated SQL: SELECT name FROM sqlite_master WHERE type='table';

Final Result:
 [('users',)]


In [41]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT * FROM users"))
    print(result.fetchall())

[(1, 'Ravi', 30), (2, 'Anil', 25), (3, 'Sita', 28)]


In [40]:
app_graph = graph.compile()
run_query("what is your name ?")

RAW LLM OUTPUT: sql
FINAL ROUTE: sql
→ SQL Agent
Generated SQL: SELECT name FROM users WHERE id = 1;

Final Result:
 [('Ravi',)]


In [42]:
docs = [
    Document(page_content="RAG stands for Retrieval-Augmented Generation. It combines retrieval and LLMs."),
    Document(page_content="LangGraph is used for building agent workflows."),
    Document(page_content="Kubernetes is used for container orchestration.")
]

In [44]:
from multiprocessing import context
from winreg import QueryValue


prompt = f"""
You MUST answer ONLY using the provided context.
If answer is not in context, say "I don't know".

Context:
{context}

Question: {QueryValue}
"""

ModuleNotFoundError: No module named 'winreg'